In [ ]:
from pathlib import Path
import logging

# ─── PATHS ───────────────────────────────────────────────────────────────────
# You will manually update this path each time you switch sessions
SOURCE_DIR  = Path("/content/drive/MyDrive/RCC-hdf5/extracted_pids-3/")          
OUTPUT_DIR  = Path("/content/drive/MyDrive/ozyegin/CSV/")             
OUTPUT_CSV  = OUTPUT_DIR / "rcc_features.csv" # Change name based on session (e.g., aml_features.csv)

# ─── PHASE ───────────────────────────────────────────────────────────────────
PHASE         = 'arterial'          # HDF5 dataset key for the CT volume
PHASE_PIXDIM  = 'arterial_pixdim'  # HDF5 attribute key for that phase's spacing

# ─── MASK ────────────────────────────────────────────────────────────────────
# Kept because both your RCC and AML HDF5 containers use 1.0 for the tumor voxels
MASK_LABEL    = 1    

# ─── PYRADIOMICS EXTRACTOR SETTINGS ──────────────────────────────────────────
EXTRACTOR_SETTINGS = {

    # ── Interpolation ────────────────────────────────────────────────────────
    'interpolator'             : 'sitkBSpline',    
    'resampledPixelSpacing'    : [1, 1, 1],         
    'padDistance'              : 10,                

    # ── Mask validation ──────────────────────────────────────────────────────
    'label'                    : MASK_LABEL,        
    'minimumROIDimensions'     : 2,                 
    'minimumROISize'           : 50,                

    # ── Intensity normalisation ───────────────────────────────────────────────
    'binWidth'                 : 5,

    # ── Image pre-processing ─────────────────────────────────────────────────
    'normalize'                : True,   
    'normalizeScale'           : 100,
    'removeOutliers'           : None,    

    # ── Filters (image types) ────────────────────────────────────────────────
    'enabledImageTypes' : {
        'Original' : {},
        'LoG'      : {'sigma': [1.0, 2.0, 3.0]},   
        'Wavelet'  : {},
    },

    # ── Feature classes ───────────────────────────────────────────────────────
    'enabledFeatures' : {
        'shape'         : [],   
        'firstorder'    : [],   
        'glcm'          : [],   
        'glrlm'         : [],   
        'glszm'         : [],   
        'gldm'          : [],   
        'ngtdm'         : [],   
    },

    # ── GLCM-specific ────────────────────────────────────────────────────────
    'weightingNorm'            : None,   

    # ── Verbosity ────────────────────────────────────────────────────────────
    'verbosity'                : logging.WARNING,
}

print("Configuration loaded.")
print(f"  Source : {SOURCE_DIR}")
print(f"  Output : {OUTPUT_CSV}")
print(f"  Phase  : {PHASE}")

In [ ]:
SOURCE_DIR          = Path("/content/drive/MyDrive/RCC-hdf5/extracted_pids-3")
OUTPUT_DIR          = Path("/content/drive/MyDrive/ozyegin/CSV/")
PHASE               = "arterial"
PERTURBATION_FACTOR = 0.30

all_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith(".hdf5")])
print(f"Total HDF5 files found: {len(all_files)}")
print(f"First 5: {all_files[:5]}")

In [ ]:
def build_extractor(settings: dict) -> featureextractor.RadiomicsFeatureExtractor:
    """
    Construct a PyRadiomics extractor from a plain Python dict.
    Separates top-level params, enabledImageTypes, and enabledFeatures
    so they can each be edited as readable dictionary entries above.
    """
    enabled_image_types = settings.pop('enabledImageTypes', {'Original': {}})
    enabled_features    = settings.pop('enabledFeatures',  {})

    extractor = featureextractor.RadiomicsFeatureExtractor(**settings)

    # Disable everything first, then enable only what we want
    extractor.disableAllImageTypes()
    for img_type, params in enabled_image_types.items():
        extractor.enableImageTypeByName(img_type, customArgs=params if params else None)

    extractor.disableAllFeatures()
    for feature_class, features in enabled_features.items():
        if features:
            extractor.enableFeaturesByName(**{feature_class: features})
        else:
            extractor.enableFeatureClassByName(feature_class)

    # Restore the dicts (so config cell can be re-run safely)
    settings['enabledImageTypes'] = enabled_image_types
    settings['enabledFeatures']   = enabled_features

    return extractor


extractor = build_extractor(EXTRACTOR_SETTINGS)
print("Extractor ready.")
print(f"  Active image types : {list(extractor.enabledImagetypes.keys())}")
print(f"  Active features    : {list(extractor.enabledFeatures.keys())}")

In [ ]:
import h5py
import numpy as np
import SimpleITK as sitk
from pathlib import Path

def hdf5_volume_to_sitk(array: np.ndarray, pixdim: np.ndarray) -> sitk.Image:
    """
    Convert HDF5 dataset array (X, Y, Z) + pixdim [dx, dy, dz]
    into a SimpleITK Image with correct physical spacing.
    """
    arr_zyx = np.transpose(array, (2, 0, 1)).astype(np.float32)  # (Z, X, Y)
    img = sitk.GetImageFromArray(arr_zyx)
    img.SetSpacing([float(pixdim[0]), float(pixdim[1]), float(pixdim[2])])
    img.SetOrigin([0.0, 0.0, 0.0])
    img.SetDirection([1, 0, 0,
                      0, 1, 0,
                      0, 0, 1])
    return img


def hdf5_mask_to_sitk(array: np.ndarray, pixdim: np.ndarray) -> sitk.Image:
    """
    Convert binary mask array (X, Y, Z) to SimpleITK Image.
    Casts to uint8 — required by PyRadiomics mask validator.
    """
    arr_zyx = np.transpose(array, (2, 0, 1)).astype(np.uint8)
    img = sitk.GetImageFromArray(arr_zyx)
    img.SetSpacing([float(pixdim[0]), float(pixdim[1]), float(pixdim[2])])
    img.SetOrigin([0.0, 0.0, 0.0])
    img.SetDirection([1, 0, 0,
                      0, 1, 0,
                      0, 0, 1])
    return img


def load_case(hdf5_path: Path, phase: str, phase_pixdim_key: str):
    """
    Load a single HDF5 case.
    Returns (ct_sitk, mask_sitk, meta_dict) or raises ValueError with reason.
    Works identically for original and resampled files.
    """
    with h5py.File(hdf5_path, 'r') as f:

        # ── Validate required datasets ────────────────────────────────────────
        if phase not in f:
            raise ValueError(f"Phase '{phase}' not found in file")
        if 'mask' not in f:
            raise ValueError("'mask' dataset not found in file")

        # ── Validate required attributes ──────────────────────────────────────
        if phase_pixdim_key not in f.attrs:
            raise ValueError(f"Attribute '{phase_pixdim_key}' not found")
        if 'mask_pixdim' not in f.attrs:
            raise ValueError("Attribute 'mask_pixdim' not found")

        # ── Read volumes ──────────────────────────────────────────────────────
        ct_array    = f[phase][()]           # (512, 512, N)
        mask_array  = f['mask'][()]          # (512, 512, N)
        ct_pixdim   = np.array(f.attrs[phase_pixdim_key])
        mask_pixdim = np.array(f.attrs['mask_pixdim'])

        # ── Validate shape alignment ──────────────────────────────────────────
        if ct_array.shape != mask_array.shape:
            raise ValueError(
                f"Shape mismatch: {phase} {ct_array.shape} vs mask {mask_array.shape}. "
                f"Run the resampling notebook first."
            )

        # ── Collect metadata ──────────────────────────────────────────────────
        meta = {
            'PID'              : str(f.attrs.get('PID', hdf5_path.stem)),
            'tumor_type'       : f.attrs.get('tumor_type', -1),
            
            # FIXED: Removed LABEL_MAP lookups to bypass target class dictionary switching
            'label'            : str(f.attrs.get('label', 'Unknown')),
            
            'pathology'        : str(f.attrs.get('pathology', 'unknown')),
            'pathology_grade'  : str(f.attrs.get('pathology_grade', 'nan')),
            'Patient_Age'      : str(f.attrs.get('Patient Age', 'unknown')),
            'Patient_Sex'      : str(f.attrs.get('Patient Sex', 'unknown')),
            'Manufacturer'     : str(f.attrs.get('Manufacturer', 'unknown')),
            'phase'            : phase,
            'ct_spacing_mm'    : ct_pixdim.tolist(),
            'mask_spacing_mm'  : mask_pixdim.tolist(),
            'ct_shape'         : list(ct_array.shape),
            'mask_voxel_count' : int((mask_array == MASK_LABEL).sum()),
        }

    ct_sitk   = hdf5_volume_to_sitk(ct_array,   ct_pixdim)
    mask_sitk = hdf5_mask_to_sitk(mask_array, mask_pixdim)

    return ct_sitk, mask_sitk, meta


print("HDF5 → SimpleITK utilities loaded.")

In [ ]:
import h5py

# Scan source directory for files
all_files = sorted(SOURCE_DIR.glob('*.hdf5'))
print(f"Found {len(all_files)} HDF5 files in source directory.")

# Arrays to keep track of valid vs structurally invalid files
accepted_files = []
skipped_structure = []

for fpath in all_files:
    try:
        with h5py.File(fpath, 'r') as f:
            has_phase  = PHASE in f
            has_mask   = 'mask' in f

        # Check structural file readiness
        if not has_phase:
            skipped_structure.append((fpath.name, f'missing phase: {PHASE}'))
        elif not has_mask:
            skipped_structure.append((fpath.name, 'missing mask'))
        else:
            accepted_files.append(fpath)
            
    except Exception as e:
        # This will catch and flag corrupted files like UDBvBoHG3i automatically!
        skipped_structure.append((fpath.name, f'read error/corrupted: {e}'))

# Console Feedback
print(f"\n  Accepted (has {PHASE} + mask)      : {len(accepted_files)}")
print(f"  Skipped (missing dataset/corrupt)  : {len(skipped_structure)}")

if skipped_structure:
    print("\nSkipped Files Log:")
    for name, reason in skipped_structure:
        print(f"  {name}  → {reason}")

In [ ]:
results   = []   # list of dicts, one per patient
failed    = []   # list of (filename, error_message)

for hdf5_path in tqdm(accepted_files, desc=f'Extracting [{PHASE}] features'):

    try:
        # ── 1. Load case ─────────────────────────────────────────────────────
        ct_sitk, mask_sitk, meta = load_case(
            hdf5_path     = hdf5_path,
            phase         = PHASE,
            phase_pixdim_key = PHASE_PIXDIM,
        )

        # ── 2. Extract features ───────────────────────────────────────────────
        feature_vector = extractor.execute(ct_sitk, mask_sitk, label=MASK_LABEL)

        # ── 3. Separate diagnostics from actual features ──────────────────────
        # PyRadiomics returns diagnostic_ keys (internal QC) alongside features.
        # We keep them prefixed for transparency but put metadata columns first.
        features_only     = {k: float(v) if hasattr(v, '__float__') else v
                             for k, v in feature_vector.items()
                             if not k.startswith('diagnostics_')}

        diagnostics_only  = {k: str(v)
                             for k, v in feature_vector.items()
                             if k.startswith('diagnostics_')}

        # ── 4. Build output row ───────────────────────────────────────────────
        row = {**meta, **features_only, **diagnostics_only}
        results.append(row)

    except Exception as e:
        failed.append((hdf5_path.name, str(e)))
        tqdm.write(f"  FAILED: {hdf5_path.name} — {e}")


print(f"\nExtraction complete.")
print(f"  Successful : {len(results)}")
print(f"  Failed     : {len(failed)}")

if failed:
    print("\nFailed cases:")
    for fname, reason in failed:
        print(f"  {fname}: {reason}")

In [ ]:
if not results:
    print("No results to save — check errors above.")
else:
    df = pd.DataFrame(results)

    # ── Reorder: metadata columns first, then features, then diagnostics ──────
    meta_cols  = ['PID', 'label', 'tumor_type', 'pathology', 'pathology_grade',
                  'Patient_Age', 'Patient_Sex', 'Manufacturer',
                  'phase', 'ct_spacing_mm', 'mask_spacing_mm',
                  'ct_shape', 'mask_voxel_count']
    diag_cols  = [c for c in df.columns if c.startswith('diagnostics_')]
    feat_cols  = [c for c in df.columns if c not in meta_cols + diag_cols]

    df = df[meta_cols + feat_cols + diag_cols]

    # ── Save ──────────────────────────────────────────────────────────────────
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"CSV saved: {OUTPUT_CSV}")
    print(f"  Rows (patients) : {len(df)}")
    print(f"  Metadata columns: {len(meta_cols)}")
    print(f"  Feature columns : {len(feat_cols)}")
    print(f"  Diagnostic cols : {len(diag_cols)}")
    print(f"  Total columns   : {df.shape[1]}")
    print(f"\nLabel distribution:")
    print(df['label'].value_counts().to_string())

In [ ]:
if results:
    feat_df = df[feat_cols].apply(pd.to_numeric, errors='coerce')

    nan_counts = feat_df.isna().sum()
    nan_cols   = nan_counts[nan_counts > 0]

    print(f"Feature columns with NaN values: {len(nan_cols)}")
    if not nan_cols.empty:
        print(nan_cols.to_string())
    else:
        print("  None — all features extracted cleanly.")

    print(f"\nFirst-order feature summary (Original image):")
    fo_cols = [c for c in feat_cols if 'original_firstorder' in c]
    if fo_cols:
        print(df.groupby('label')[fo_cols[:6]].mean().round(2).to_string())